# SAM 2 Segmentation — HummingBird2

**Runtime:** Runtime > Change runtime type > **T4 GPU**

**Before running, upload these files to `MyDrive/Iridescence Lab/HummingBird2/`:**
1. The video file (`HummingBird2_1.mov`)
2. `keyframes.json`
3. `annotation_masks.npz` (the painted masks — from the Annotate tab)

**After running:** Download `masks.npz` from Drive and upload it in the app (Segment tab).

In [ ]:
# ── Install SAM 2 ──────────────────────────────────────────────────────────
!pip install -q git+https://github.com/facebookresearch/sam2.git

import os
if not os.path.exists('/content/sam2_hiera_small.pt'):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt -P /content/
    print('Checkpoint downloaded.')
else:
    print('Checkpoint already present.')

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import cv2, json, gc, os, shutil
import numpy as np
import torch
from pathlib import Path
from contextlib import nullcontext

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}  |  device: {device}')

def autocast_ctx(dev):
    return torch.autocast('cuda', dtype=torch.bfloat16) if dev == 'cuda' else nullcontext()

In [ ]:
# ── Mount Drive ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Paths — adjust DRIVE_FOLDER if your Drive layout differs ──────────────
SPECIES         = 'HummingBird2'
DRIVE_FOLDER    = '/content/drive/MyDrive/Iridescence Lab/HummingBird2'
VIDEO_PATH      = DRIVE_FOLDER + '/HummingBird2_1.mov'
KF_PATH         = DRIVE_FOLDER + '/keyframes.json'
ANN_MASKS_PATH  = DRIVE_FOLDER + '/annotation_masks.npz'
MASKS_OUT       = DRIVE_FOLDER + '/masks.npz'

assert os.path.exists(VIDEO_PATH), 'Video not found: ' + VIDEO_PATH
assert os.path.exists(KF_PATH),    'keyframes.json not found: ' + KF_PATH
print('Ready:', SPECIES)
print('Painted masks available:', os.path.exists(ANN_MASKS_PATH))

In [ ]:
# ── Extract frames ──────────────────────────────────────────────────────────
FRAMES_DIR = '/content/frames'
os.makedirs(FRAMES_DIR, exist_ok=True)

cap    = cv2.VideoCapture(VIDEO_PATH)
fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'{width}x{height} @ {fps:.1f} FPS — {total} frames ({total/fps:.1f}s)')

saved = 0
while True:
    ret, frame = cap.read()
    if not ret: break
    cv2.imwrite(f'{FRAMES_DIR}/{saved:05d}.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
    saved += 1
cap.release()
print(f'Extracted {saved} frames.')

In [ ]:
# ── Load keyframes + painted annotation masks ────────────────────────────────
import json as _json
with open(KF_PATH) as f:
    KEYFRAMES = _json.load(f)

all_parts = []
for kf in KEYFRAMES:
    for k in kf:
        if k != 'frame' and k not in all_parts:
            all_parts.append(k)

PART_IDS   = {part: i+1 for i, part in enumerate(all_parts)}
ID_TO_PART = {v: k for k, v in PART_IDS.items()}

# Load painted masks (preferred over point prompts)
ANN_MASKS = {}
if os.path.exists(ANN_MASKS_PATH):
    _d = np.load(ANN_MASKS_PATH)
    ANN_MASKS = {k: _d[k] for k in _d.files}
    print(f'Loaded {len(ANN_MASKS)} painted mask entries')
else:
    print('No annotation_masks.npz found — using point prompts (less precise)')

print('Parts:', PART_IDS)
print(f'Loaded {len(KEYFRAMES)} keyframes')

In [ ]:
# ── Load SAM 2 ──────────────────────────────────────────────────────────────
from sam2.build_sam import build_sam2_video_predictor

predictor = build_sam2_video_predictor(
    config_file='sam2_hiera_s.yaml',
    ckpt_path='/content/sam2_hiera_small.pt',
    device=device,
)
print(f'SAM 2 loaded on {device}.')

In [ ]:
# ── Chunked propagation (300 frames/chunk to stay within Colab RAM) ─────────
CHUNK_SIZE = 300
CHUNK_DIR  = '/content/frames_chunk'

kf_by_frame = {kf['frame']: kf for kf in KEYFRAMES}
all_frames  = sorted(int(f.stem) for f in Path(FRAMES_DIR).glob('*.jpg'))
chunks      = [all_frames[i:i+CHUNK_SIZE] for i in range(0, len(all_frames), CHUNK_SIZE)]

video_segments = {}
carry_masks    = None

print(f'{len(all_frames)} frames  ->  {len(chunks)} chunks of <={CHUNK_SIZE}\n')

for ci, chunk in enumerate(chunks):
    chunk_start = chunk[0]
    chunk_kfs   = [f for f in chunk if f in kf_by_frame]
    anchor_orig = chunk_kfs[0] if chunk_kfs else chunk[0]
    anchor_rel  = anchor_orig - chunk_start

    label = f'keyframes {chunk_kfs}' if chunk_kfs else 'carry-over'
    print(f'Chunk {ci+1}/{len(chunks)}: frames {chunk[0]}-{chunk[-1]}  ({label})')

    tmp = Path(CHUNK_DIR)
    if tmp.exists(): shutil.rmtree(tmp)
    tmp.mkdir()
    for i, fidx in enumerate(chunk):
        os.symlink(f'{FRAMES_DIR}/{fidx:05d}.jpg', f'{CHUNK_DIR}/{i:05d}.jpg')

    with torch.inference_mode(), autocast_ctx(device):
        state = predictor.init_state(video_path=CHUNK_DIR, offload_video_to_cpu=True)
        predictor.reset_state(state)
        prompts_added = False

        if chunk_kfs:
            for fidx in chunk_kfs:
                kf  = kf_by_frame[fidx]
                rel = fidx - chunk_start
                for part, obj_id in PART_IDS.items():
                    val      = kf.get(part)
                    mask_key = f'{fidx}_{part}'
                    if val in (None, 'skip'):
                        # Part not visible — negative center point
                        predictor.add_new_points_or_box(
                            inference_state=state, frame_idx=rel,
                            obj_id=obj_id,
                            points=np.array(
                                [[width//2, height//2]], dtype=np.float32),
                            labels=np.array([0], dtype=np.int32),
                        )
                    elif mask_key in ANN_MASKS:
                        # Painted mask prompt — most precise
                        predictor.add_new_mask(
                            inference_state=state, frame_idx=rel,
                            obj_id=obj_id,
                            mask=torch.from_numpy(
                                ANN_MASKS[mask_key].astype(bool)),
                        )
                    elif isinstance(val, list):
                        # Legacy point prompt (backward compat)
                        predictor.add_new_points_or_box(
                            inference_state=state, frame_idx=rel,
                            obj_id=obj_id,
                            points=np.array([val], dtype=np.float32),
                            labels=np.array([1], dtype=np.int32),
                        )
            prompts_added = True

        elif carry_masks is not None:
            for part, obj_id in PART_IDS.items():
                mask = carry_masks.get(part)
                if mask is not None and mask.any():
                    predictor.add_new_mask(
                        inference_state=state, frame_idx=0, obj_id=obj_id,
                        mask=torch.from_numpy(mask),
                    )
                    prompts_added = True

        if not prompts_added:
            print('  -> no subject visible, skipping.')
            predictor.reset_state(state); del state
            gc.collect(); torch.cuda.empty_cache()
            shutil.rmtree(CHUNK_DIR)
            carry_masks = None
            continue

        for rel, obj_ids, logits in predictor.propagate_in_video(
                state, start_frame_idx=anchor_rel):
            orig = chunk_start + rel
            video_segments[orig] = {
                ID_TO_PART[oid]: (logits[i] > 0.0).cpu().numpy().astype(bool).squeeze()
                for i, oid in enumerate(obj_ids)
            }
            del logits

        if anchor_rel > 0:
            for rel, obj_ids, logits in predictor.propagate_in_video(
                    state, start_frame_idx=anchor_rel, reverse=True):
                orig = chunk_start + rel
                video_segments[orig] = {
                    ID_TO_PART[oid]: (logits[i] > 0.0).cpu().numpy().astype(bool).squeeze()
                    for i, oid in enumerate(obj_ids)
                }
                del logits

        last_orig = chunk[-1]
        if last_orig in video_segments:
            last_masks = video_segments[last_orig]
            carry_masks = (
                {p: last_masks[p].copy() for p in last_masks}
                if any(m.any() for m in last_masks.values()) else None
            )

        predictor.reset_state(state); del state

    gc.collect(); torch.cuda.empty_cache(); shutil.rmtree(CHUNK_DIR)
    n_done = sum(1 for f in chunk if f in video_segments)
    print(f'  -> {n_done}/{len(chunk)} frames done  |  total: {len(video_segments)}')

print(f'\nDone: {len(video_segments)} / {len(all_frames)} frames segmented.')

In [ ]:
# ── Save masks to Google Drive ──────────────────────────────────────────────
mask_dict = {}
for fidx, parts in video_segments.items():
    for part, mask in parts.items():
        mask_dict[f'{fidx:05d}_{part}'] = mask

mask_dict['_fps']       = np.array([fps])
mask_dict['_frame_wh']  = np.array([width, height])
mask_dict['_ann_frame'] = np.array([KEYFRAMES[0]['frame']])

np.savez_compressed(MASKS_OUT, **mask_dict)
size_mb = os.path.getsize(MASKS_OUT) / 1e6
print(f'Saved {len(video_segments)} frames -> {MASKS_OUT}')
print(f'File size: {size_mb:.1f} MB')
print()
print('Download masks.npz from Google Drive, then upload it in the app (Segment tab).')